# BMC - GBIF

The BMC library offers a simplified API with the GBIF API. This simplified functionality is presented in the `datasource.gbif` module containting the submodules `interface` and `sql`.

1) `interface` provides a simplified way to handle taxonomic matching and abstract a lot of `pygbif` behaviour. The main focus of this submodule is the `fetch_taxon_info` function which can be used to perform name matching between a list of names directly to the GBIF or COL taxonomic identifiers
2) `sql` provides a simplified way of querying GBIF data using the SQL API. The main focus of this submodule is the `generate_sql_query` function that generates SQL ready strings used to query taxonomy, space and time according to their specifications.

Using the `datasource` we implement a standardized processing workflow for the `gbif_cube`. GBIF data generally is queried using either point data or pregridded information according to the grids defined within the SQL API. As such the resulting data that is returned from the database is vector data and as such `gbif_cube` is a child of the `spatiotemporal_vector_cube` class. The implemented workflow processes either the processed gridded data or the raw point data to be aligned with the other cubes produced in the engine

## GBIF-Datasource

## `interface`

### `fetch_taxon_info`

#### Overview

The central function of the `interface` that the user has to handle is the `fetch_taxon_info`. Upon inspecting the function we see that the goal of the function is to perform matching to either the GBIF or COL backbone and give an overview of matching type and the associated key for the queried names back to the user. The end result of the function is 
1) `taxonomic_df`, containing all the matches that have a 1-1 mapping in the taxonomic backbone of interest
2) `mismatch_df`, containing all the matches that can not be resolved exactly. By default this also includes fuzzy, variant and higherrank matches

In [1]:
from bmc.datasource.gbif import interface
interface.fetch_taxon_info?

KeyboardInterrupt: 

#### Notes on GBIF matchType

In the Global Biodiversity Information Facility (GBIF) database, the `matchType` field is a classification flag returned by the taxonomic name matching API (such as the Species Match service). It indicates exactly how a user-provided scientific name (`verbatimScientificName`) was resolved against the authoritative GBIF Backbone Taxonomy.

The `matchType` field uses a controlled vocabulary with four possible values:

| `matchType` Value | Description | Common Triggers & Notes |
| :--- | :--- | :--- |
| **`EXACT`** | A perfect string match to a single taxon in the GBIF backbone. | The binomial or polynomial spelling was exactly correct. *Note: This primarily focuses on the name string and may ignore authorship information to make the match.* |
| **`FUZZY`** | A non-exact, probabilistic match. | The API matched the supplied name assuming a slight misspelling, typographical error, or orthographic variant. |
| **`HIGHERRANK`** | Matched to a higher taxonomic grouping (e.g., Genus, Family) rather than the specific rank provided. | 1. The specific species name is missing from the backbone.<br>2. The input intrinsically represents a higher rank.<br>3. **Ambiguity:** Multiple identically spelled names exist (homonyms) and missing authorship forced the system to default to the nearest unambiguous higher rank. |
| **`NONE`** | No match could be made anywhere in the GBIF taxonomic backbone. | Severely malformed names, unmatchable strings (e.g., "Unknown species"), or obscure names completely absent from GBIF's source checklists. |

#### Notes on GBIF `status`

While the `matchType` field describes *how* your input string was matched to the database, the `status` field describes the **taxonomic validity** of the name that was matched. 

When a name is matched, GBIF returns its current standing in the taxonomic backbone. If a name is not the accepted term (e.g., a synonym), the API response will usually provide an `acceptedUsageKey` and `accepted` name string to point you to the correct, currently valid taxon.

The `status` field uses a controlled vocabulary. Below are the primary values you will encounter:

| `status` Value | Description | Notes & Interpretation |
| :--- | :--- | :--- |
| **`ACCEPTED`** | The name is recognized as a currently valid and correct scientific name. | This is the target status for most modern biodiversity data. No further taxonomic resolution is needed. |
| **`SYNONYM`** | The name is recognized as a synonym of another accepted taxon. | General synonym category. GBIF will redirect you to the currently accepted name via the `acceptedUsageKey`. |
| **`DOUBTFUL`** | The validity or application of the name is uncertain or questionable. | Often corresponds to a *nomen dubium*. It represents a name that cannot be assigned to a specific taxon with certainty. |
| **`HETEROTYPIC_SYNONYM`** | A taxonomic (or subjective) synonym. | The synonym and the accepted name are based on **different** type specimens, but taxonomists currently consider them to be the same species. |
| **`HOMOTYPIC_SYNONYM`** | A nomenclatural (or objective) synonym. | The synonym and the accepted name are based on the **exact same** type specimen (e.g., a species was moved to a new genus). |
| **`PROPARTE_SYNONYM`** | A "partial" synonym (*pro parte*). | The original name was applied broadly and has since been split. The synonym applies to only a part of the original taxon's circumscription. |
| **`MISAPPLIED`** | The name was incorrectly applied to a different taxon by an author. | Often found in historical literature where a valid name was mistakenly used to describe a completely different species. |

> **Pro Tip for Data Cleaning:** When resolving taxa, any record returning a `status` other than `ACCEPTED` (or occasionally `DOUBTFUL`) should be updated to use the provided accepted name and accepted taxon key to ensure standardized data alignment.

#### Examples

##### List of exact names

For the list of names that we will use throughout the tutorial we will request information regarding the Asian hornet and some of its established predators. This list has already been vetted and will produce a `taxonomic_df` containing all the names and an empty `mismatch_df` due to feeding the function a well processed list

In [ ]:
vespa_velutina_predators = [
    "Pernis apivorus",
    "Dendrocopos major",
    "Parus major",
    "Garrulus glandarius",
    "Pica pica",
    "Merops apiaster",
    "Meles meles",
    "Martes martes",
    "Martes foina",
    "Vespa crabro",
    "Argiope bruennichi",
    "Araneus diadematus"
]
taxonInfo, _ = interface.fetch_taxon_info(["Vespa velutina"]+vespa_velutina_predators)
taxonInfo

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames
0,EXACT,2160133,2160133,"Araneus diadematus Clerck, 1757",SPECIES,ACCEPTED,Araneus diadematus
1,EXACT,2480420,2480420,"Pernis apivorus (Linnaeus, 1758)",SPECIES,ACCEPTED,Pernis apivorus
2,EXACT,5218878,5218878,"Martes martes (Linnaeus, 1758)",SPECIES,ACCEPTED,Martes martes
3,EXACT,1311527,1311527,Vespa crabro Linnaeus,SPECIES,ACCEPTED,Vespa crabro
4,EXACT,5229490,5229490,"Pica pica (Linnaeus, 1758)",SPECIES,ACCEPTED,Pica pica
5,EXACT,5218887,5218887,"Martes foina (Erxleben, 1777)",SPECIES,ACCEPTED,Martes foina
6,EXACT,9705453,9705453,"Parus major Linnaeus, 1758",SPECIES,ACCEPTED,Parus major
7,EXACT,2475443,2475443,"Merops apiaster Linnaeus, 1758",SPECIES,ACCEPTED,Merops apiaster
8,EXACT,2433875,2433875,"Meles meles (Linnaeus, 1758)",SPECIES,ACCEPTED,Meles meles
9,EXACT,2477968,2477968,"Dendrocopos major (Linnaeus, 1758)",SPECIES,ACCEPTED,Dendrocopos major


As is clear in the table, all `matchType` entries are `EXACT` and the `status` entries are `ACCEPTED` and as such this represents a correct and complete list. The `mismatch_df` returned is an empty df

In [ ]:
_

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames


The dataframe was above was generated against the GBIF taxonomic backbone. Optionally we can also generate the equivalent table using the COL identfiers by including the optional argument `col_backbone=True`

In [ ]:
taxonInfo, _ = interface.fetch_taxon_info(["Vespa velutina"]+vespa_velutina_predators, col_backbone=True)
taxonInfo

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames
0,EXACT,G59D,G59D,"Araneus diadematus Clerck, 1757",SPECIES,ACCEPTED,Araneus diadematus
1,EXACT,4F6YZ,4F6YZ,"Pernis apivorus (Linnaeus, 1758)",SPECIES,ACCEPTED,Pernis apivorus
2,EXACT,3Y9W2,3Y9W2,"Martes martes (Linnaeus, 1758)",SPECIES,ACCEPTED,Martes martes
3,EXACT,5B9T3,5B9T3,Vespa crabro Linnaeus,SPECIES,ACCEPTED,Vespa crabro
4,EXACT,4HPXM,4HPXM,"Pica pica (Linnaeus, 1758)",SPECIES,ACCEPTED,Pica pica
5,EXACT,3Y9VW,3Y9VW,"Martes foina (Erxleben, 1777)",SPECIES,ACCEPTED,Martes foina
6,EXACT,75SVV,75SVV,"Parus major Linnaeus, 1758",SPECIES,ACCEPTED,Parus major
7,EXACT,3ZW55,3ZW55,"Merops apiaster Linnaeus, 1758",SPECIES,ACCEPTED,Merops apiaster
8,EXACT,3ZDRX,3ZDRX,"Meles meles (Linnaeus, 1758)",SPECIES,ACCEPTED,Meles meles
9,EXACT,6CJDB,6CJDB,"Dendrocopos major (Linnaeus, 1758)",SPECIES,ACCEPTED,Dendrocopos major


##### List containing errors

We will now illustrate the result of a list containing names that do not resolve properly in the API. We will include a mix of different types of wrong matches and how to deal with them appropriately. The list wil be as follows


| Input String (`verbatimScientificName`) | `matchType` | `status` | Resolved GBIF Name | Explanation |
| :--- | :--- | :--- | :--- | :--- |
| **`Vulpes vulpes`** | `EXACT` | `ACCEPTED` | *Vulpes vulpes* | The perfect, accepted spelling of the Red Fox. It matches directly to a valid species. |
| **`Felis concolor`** | `EXACT` | `HOMOTYPIC_SYNONYM` | *Puma concolor* | Perfectly spelled, but taxonomically outdated. It is an exact string match to a known synonym, which points to the accepted name *Puma concolor*. |
| **`Pantera leo`** | `FUZZY` | `ACCEPTED` | *Panthera leo* | The genus is missing an 'h'. The API catches the typo, makes a fuzzy match, and successfully lands on the accepted lion species. |
| **`Fellis concolor`** | `FUZZY` | `HOMOTYPIC_SYNONYM` | *Puma concolor* | A double-layer correction. The API fixes the typo ("Fellis" to "Felis") via a fuzzy match, realizes *Felis concolor* is a synonym, and points to *Puma concolor*. |
| **`Glocianus punctiger`** | `HIGHERRANK` | `ACCEPTED` | *Glocianus* (Genus) | **Homonym conflict.** Two different authors described a beetle named *Glocianus punctiger*. Because no authorship was provided in the input, the API cannot safely guess which one is meant, so it defaults to matching at the Genus level. |
| **`Canis crypticus`** | `HIGHERRANK` | `ACCEPTED` | *Canis* (Genus) | A fictional or completely undocumented species name. The API recognizes the valid genus *Canis*, but drops the unrecognized specific epithet. |
| **`Eohippus crypticus`** | `HIGHERRANK` | `SYNONYM` | *Hyracotherium* (Genus) | Fictional species placed in an invalid genus. The API matches the genus *Eohippus*, recognizes that genus is a synonym of *Hyracotherium*, and defaults the match to that accepted genus. |
| **`Trachodon mirabilis`** | `EXACT` | `DOUBTFUL` | *Trachodon mirabilis* | Perfectly spelled, but it is a *nomen dubium* (based on fossil teeth that cannot be confidently assigned to a specific modern dinosaur lineage). The name exists, but its application is uncertain. |
| **`Unknown insect sp.`** | `NONE` | *N/A* | *None* | A completely unmatchable string with no recognizable taxonomic structure in the GBIF backbone. |

In [ ]:
gbif_test_names = [
    "Vulpes vulpes",
    "Felis concolor",
    "Pantera leo",
    "Fellis concolor",
    "Glocianus punctiger",
    "Canis crypticus",
    "Eohippus crypticus",
    "Trachodon mirabilis",
    "Unknown insect sp."
]

taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names)
taxonInfo_errors

Mismatches encountered of type {'HIGHERRANK', 'VARIANT', 'EXACT', 'NONE'}
Count HIGHERRANK: 3
Count VARIANT: 2
Count EXACT: 3
Count NONE: 1


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5219243.0,5219243,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
5,EXACT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
7,EXACT,8573909.0,8573909,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN


In [ ]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
1,VARIANT,5219404.0,5219404.0,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,HIGHERRANK,5219142.0,5219142.0,"Canis Linnaeus, 1758",GENUS,ACCEPTED,Canis crypticus,NaN
3,VARIANT,2435104.0,2435099.0,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No GBIF match
6,HIGHERRANK,4239.0,4239.0,Curculionidae,FAMILY,ACCEPTED,Glocianus punctiger,NaN
8,HIGHERRANK,4830484.0,4830484.0,"Eohippus Marsh, 1876",GENUS,ACCEPTED,Eohippus crypticus,NaN


Based on the mismatches and the `note` and `matchType` the user can manually review how to proceed. If they wish to include `FUZZY` and `HIGHERRANK` matches into their they can toggle this in the function argument by using `keep_fuzzy` and `keep_higherrank` 

In [ ]:
taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names, keep_fuzzy=True, keep_higherrank=True)
taxonInfo_errors

Mismatches encountered of type {'HIGHERRANK', 'VARIANT', 'EXACT', 'NONE'}
Count HIGHERRANK: 3
Count VARIANT: 2
Count EXACT: 3
Count NONE: 1


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5219243.0,5219243,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
1,VARIANT,5219404.0,5219404,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,HIGHERRANK,5219142.0,5219142,"Canis Linnaeus, 1758",GENUS,ACCEPTED,Canis crypticus,NaN
3,VARIANT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
5,EXACT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
6,HIGHERRANK,4239.0,4239,Curculionidae,FAMILY,ACCEPTED,Glocianus punctiger,NaN
7,EXACT,8573909.0,8573909,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN
8,HIGHERRANK,4830484.0,4830484,"Eohippus Marsh, 1876",GENUS,ACCEPTED,Eohippus crypticus,NaN


In [ ]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No GBIF match


Note, matching in one backbone does not ensure that matching will be exactly the same in the other backbone. Depending on the status of the taxonomic backbone or to which extent it is harmonized, results might be different

In [ ]:
taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names, col_backbone=True)
taxonInfo_errors

Mismatches encountered of type {'NONE', 'EXACT', 'FUZZY'}
Count NONE: 3
Count EXACT: 4
Count FUZZY: 2


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5BSG3,5BSG3,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
5,EXACT,3DXV5,299463461,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
6,EXACT,NWPFG,NWPFG,"Glocianus punctiger (C.R. Sahlberg, 1835)",SPECIES,ACCEPTED,Glocianus punctiger,NaN
7,EXACT,TG8W8,TG8W8,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN


In [ ]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
1,FUZZY,4CGXP,4CGXP,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,NONE,NaN,NaN,NaN,NaN,NaN,Canis crypticus,No CoL match
3,FUZZY,3DXV5,299463461,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No CoL match
8,NONE,NaN,NaN,NaN,NaN,NaN,Eohippus crypticus,No CoL match


We note that `Canis crypticus` and `Eohippus crypticus` have no existing COL match and thus an alternative name must be found for it

## `sql`

Using the results from the `fetch_taxon_info` function we can now start building queries using the `generate_query` function in the `sql` submodule of the library. The information from the previous steps will be utilized as an parameter option in the cube_recipe to generate a SQL query that can either return

- raw unaggregated point data that can either be stored in its native raw format
- raw unaggregated point data that will be fed to the vector engine and spatially aligned with the other requested data from the cubing engine. The vector engine offers multiple topology types and algorithms that can geospatially process the data according to your needs. Check tutotial `0c_spatial_vector_engine.ipynb` for a demonstration and explanation of the different algorithms and topology types and tutorial `1b_spatiotemporal_vector_cube.ipynb` for a demonstration of the gbif cube itself as a fundamental example of the cubing
- GBIF SQL cubes that are cubed according to GBIFs own cubing code


In [ ]:
from bmc.datasource.gbif import sql
from bmc.cube import gbif
sql.generate_query?

Signature:
sql.generate_query(
    taxonKeys: list[int | str] | dict[str, list[int | str]],
    columns: list[str],
    record_type: str,
    wkt_polygon: str,
    year_range: int | list[int] | None = None,
    month_range: int | list[int] | None = None,
    aggregate: bool = False,
    sql_group_cols: list[str] | None = None,
    sql_metric_selects: list[str] | None = None,
    grid: str | bool = False,
    grid_resolution: int | None = None,
    coordinateUncertainty: int = 1000,
    max_uncertainty: int | float | str | None = None,
    includeUnknownStatus: bool = True,
    include_uncertainty: bool = True,
    issue_flags: list[str] | None = None,
    col_backbone: bool = False,
    col_uuid: str = '7ddf754f-d193-4cc9-b351-99906754a03b',
) -> str
Docstring:
Generate a syntactically validated SQL string optimized for the GBIF query engine.

Assembles SELECT, WHERE, and spatial or grid-based GROUP BY clauses into a 
pretty-printed, readable SQL string. Includes dynamic bounding box o

### General overview and recipe explanaition

All of these options utilize the same base of the recipe

```
base_dir: "./test_outputs/"
cube_name: "dev_cube"

export_as:
  format: "zarr"
# ======================================================================
# 1. SPATIAL & TEMPORAL SELECTION
# ======================================================================
spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox: #Flanders bbox
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900
temporal:
  start_year: 2004  #start year of the european invasion of the asian hornet
  start_month: 1
  end_year: 2025
  end_month: 12
  time_aggregation: "monthly"
```

Where we control the temporal and spatial extent in which the data is sampled. When querying from GBIF we utilize the `build_safe_fetch_envelope` function to ensure that no data starvation happens at the edges. For a visualization of what this function specifically does and how it impacts sampling consult tutorial `0a_base_spatial_grid.ipynb` for a better understanding

When it comes to requesting either raw point data vs precubed data we have to explore the options in the GBIF source field of the recipe

```
gbif:
    # =======================================================================
    # 1. QUERY & FILTER CONFIGURATION
    # Defines exactly what data is pulled from the GBIF network.
    # =======================================================================
    query_filters:
      # General taxonKey parser; accepts keys from all taxonomic levels (e.g., family, genus, species).
      taxon_keys: ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV','3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
      
      # If true, ignores taxon_keys and fetches ALL occurrence data within the Area of Interest (AOI).
      fetch_all_taxa: false
      
      # Determines the type of records to fetch. Options: "presence", "absence", or "mixed" (both).
      record_type: "presence"
      
      # Fallback coordinate uncertainty (in meters) assigned to records missing this value.
      # If left blank, defaults to matching the spatial target_resolution.
      default_Uncertainty: 1000
      
      # Hard cutoff for coordinate uncertainty. Records exceeding this radius (in meters) are dropped.
      max_uncertainty: 1000
      
      # GBIF data quality flags to exclude. Drops records flagged with these specific issues.
      exclude_issues:
        - "ZERO_COORDINATE"
        - "COORDINATE_OUT_OF_RANGE"
        - "COUNTRY_COORDINATE_MISMATCH" 

    # =======================================================================
    # 2. TAXONOMIC HARMONIZATION
    # Controls how taxonomic lineages are resolved and appended.
    # =======================================================================
    taxonomy:
      # If true, cross-references and aligns taxa against the Catalogue of Life (CoL) backbone.
      col_backbone: true
      
      # Defines the deepest taxonomic rank to include (kingdom, phylum, class, order, family, genus).
      # If left empty or removed, only the species key will be retained as an identifier in the final table.
      lowest_level: 

    # =======================================================================
    # 3. ATTRIBUTE RETENTION
    # =======================================================================
    # Additional Darwin Core columns to extract and retain alongside the geometries.
    columns: ["year", "month"]

    # =======================================================================
    # 4. SPATIAL INGESTION & PROCESSING ROUTING
    # =======================================================================
    # Defines exactly how the pipeline handles spatial geometries.
    # Options: 
    #  - "raw": Bypasses all grid math. Returns raw, unmapped occurrence records.
    #  - "vector": Downloads raw points, applies geometric topologies (buffers/clouds), and maps to target grid.
    #  - "api_cube": Uses GBIF's server-side pre-aggregated SQL grid API (fast, contracted data).
    processing_mode: "vector"

    # -----------------------------------------------------------------------
    # ROUTE A: PURE VECTOR PROCESSING (Used when processing_mode: "vector")
    # Handles raw coordinate mathematics and spatial transformations.
    # -----------------------------------------------------------------------
    vector_processing:
      # Desired geometric topology: "point", "polygon", or "point_cloud".
      topology: "point_cloud"
      
      # Mapping output: "classification" (1-to-1 cell assignment) or 
      # "fractional" (1-to-N probability/areal weights).
      mapping_mode: "fractional"  
      
      # Spatial lookup strategy: "intersect" (precise overlap) or "kdtree" (nearest centroid).
      # Note: KDTree is only mathematically valid when mapping_mode is 'classification'.
      spatial_method: "intersect"

      # Topology-specific configurations passed directly to the geometry engines.
      # The orchestrator will only apply the config matching the active 'topology'.
      topology_config:
        point_cloud:
          # Number of randomized points generated in the point cloud
          n_passes: 100
          
          # Underlying probability distribution (either "gaussian" or "uniform") 
          # with a width determined by the uncertainty radius
          distribution: "gaussian"
          
          # Random seed generator to ensure reproducibility
          random_seed: 42

        polygon:
          # Number of quadrant segments of the polygon determining the smoothness of the polygon being constructed.
          # 2-4 is very coarse, 16-32 is very smooth, 8 is the default value in many libraries.
          # Note: Increasing this will increase the impact on RAM memory during intersection operations.
          quad_segs: 8

    # -----------------------------------------------------------------------
    # ROUTE B: GBIF API CUBE CONFIGURATION (Used when processing_mode: "api_cube")
    # Maps pre-aggregated GBIF SQL grid cells using cell collection routines.
    # -----------------------------------------------------------------------
    api_cube_config:
      # "discrete" (integer sums/majority rule) or "continuous" (areal weighting).
      data_type: "discrete"       
      
      # "intersect" (geometric overlap/clipping) or "kdtree" (centroid snapping).
      spatial_method: "intersect"

    # =======================================================================
    # 5. DATA AGGREGATION & METRICS
    # =======================================================================
    aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["year", "month"]
      
      metrics:
        # 1. Expected Occurrences (Sums the geometric fractions)
        - column: "areal_fraction"
          method: "sum"
          weighted: false
          rename: "expected_occurrences"
          
        # 2. Species Count / Richness
        - column: "speciesKey"
          method: "nunique"
          weighted: false
          rename: "species_richness"
          
        # 3. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "distinct_observers"
```

In the example we will perform data generation for the taxa we processed earlier and look at the Flemish diamond region (a region spanning the 4 large flemish cities, i.e. Ghent, Antwerp, Brussels and Leuven)

### Generating raw ungridded data 

#### Using the sql function

In [ ]:
# =======================================================================
# 1. SPATIAL SETUP
# Convert the raw WGS84 bounding box into a WKT Polygon string
# =======================================================================
raw_bbox = (2.591733, 50.680797, 5.883999, 51.963900)
wkt_poly = sql.bbox2polygon_wkt(raw_bbox)

# =======================================================================
# 2. TAXONOMIC SETUP
# Because col_backbone is True, we must map these alphanumeric CoL keys 
# to their specific taxonomic rank columns before feeding them to the query
# =======================================================================
raw_taxa = ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV',
            '3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
mapped_taxa = sql.map_taxonkeys_to_columns(raw_taxa, col_backbone=True)

# =======================================================================
# 3. DATA QUALITY SETUP
# Format the exclusion flags into explicit GBIF SQL array checks
# =======================================================================
issue_flags = [
    "hasCoordinate = TRUE",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'ZERO_COORDINATE', TRUE)",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'COORDINATE_OUT_OF_RANGE', TRUE)",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'COUNTRY_COORDINATE_MISMATCH', TRUE)"
]

# =======================================================================
# 4. GENERATE THE SQL QUERY
# Pass all parameters directly to the SQL builder
# =======================================================================
manual_query = sql.generate_query(
    taxonKeys=mapped_taxa,
    columns=["year", "month", "recordedby", "speciesKey"],
    
    # "presence" in the YAML maps directly to "occurrence" in GBIF SQL
    record_type="occurrence",               
    wkt_polygon=wkt_poly,
    
    year_range=[2004, 2025],
    month_range=[1, 12],
    
    # Because processing_mode is "vector", we request raw records instead of an API Cube
    aggregate=False,                        
    grid=False,                             
    
    coordinateUncertainty=1000,
    max_uncertainty=1000,
    issue_flags=issue_flags,
    col_backbone=True
)

print(manual_query)

SELECT
    gbifid,
    "year",
    "month",
    recordedby,
    classificationdetails['7ddf754f-d193-4cc9-b351-99906754a03b']['specieskey'] AS specieskey,
    decimalLatitude,
    decimalLongitude,
    COALESCE(coordinateUncertaintyInMeters, 1000) AS coordinateUncertaintyInMeters
FROM occurrence
WHERE
    decimalLatitude >= 50.680797 AND decimalLatitude <= 51.9639
    AND decimalLongitude >= 2.591733 AND decimalLongitude <= 5.883999
    AND GBIF_Within('POLYGON ((5.883999 50.680797, 5.883999 51.9639, 2.591733 51.9639, 2.591733 50.680797, 5.883999 50.680797))', decimalLatitude, decimalLongitude) = TRUE
    AND COALESCE(coordinateUncertaintyInMeters, 1000) <= 1000
    AND "year" IS NOT NULL AND "month" IS NOT NULL
    AND "year" >= 2004 AND "year" <= 2025
    AND "month" >= 1 AND "month" <= 12
    AND hasCoordinate = TRUE
    AND NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'ZERO_COORDINATE', TRUE)
    AND NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'COORDINATE_OUT_OF_RANGE', TRUE)


#### Using the cube recipe

In [ ]:
raw_data_recipe = """
base_dir: "./gbif/"
cube_name: "gbif_cube"

export_as:
  format: "zarr"

spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox: #Flanders bbox
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900
temporal:
  start_year: 2004  #start year of the european invasion of the asian hornet
  start_month: 1
  end_year: 2025
  end_month: 12
sources:
    gbif:
        query_filters:
            taxon_keys: ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV','3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
            record_type: "presence"
            default_Uncertainty: 1000
            max_uncertainty: 1000
            exclude_issues:
                - "ZERO_COORDINATE"
                - "COORDINATE_OUT_OF_RANGE"
                - "COUNTRY_COORDINATE_MISMATCH" 
        taxonomy:
            col_backbone: true
        columns: ["year", "month", "recordedby"]
        processing_mode: "vector"
"""
# For the raw point data we can either use 'vector' or 'raw' at the processing_mode and we can drop the aggregate section of the recipe 
import yaml

# Parse the YAML string into a Python dictionary
parsed_recipe = yaml.safe_load(raw_data_recipe)

# Pass the parsed dictionary to the generator
cube = gbif.gbif_cube()
point_query = cube.generate_gbif_query_from_recipe(parsed_recipe)
print(point_query)


Constructed and resolved master grid key: EEA_1km
Computing safe fetch envelope for target grid 'EEA_1km'...
source_resolution omitted for custom CRS 'EPSG:4326'. Applying default 30 arc-second fallback: 0.008333333333333333
Safe Source Envelope (EPSG:4326): (2.34133, 50.47560, 6.03892, 52.16588)
SELECT
    gbifid,
    classificationdetails['7ddf754f-d193-4cc9-b351-99906754a03b']['specieskey'] AS specieskey,
    "year",
    "month",
    recordedby,
    decimalLatitude,
    decimalLongitude,
    COALESCE(coordinateUncertaintyInMeters, 1000) AS coordinateUncertaintyInMeters
FROM occurrence
WHERE
    decimalLatitude >= 50.47560476531819 AND decimalLatitude <= 52.16588306988871
    AND decimalLongitude >= 2.34133036280974 AND decimalLongitude <= 6.038920099374839
    AND GBIF_Within('POLYGON ((6.038920099374839 50.47560476531819, 6.038920099374839 52.16588306988871, 2.34133036280974 52.16588306988871, 2.34133036280974 50.47560476531819, 6.038920099374839 50.47560476531819))', decimalLatitu

### Generating pre gridded data

#### Using the sql function

In [ ]:
# =======================================================================
# 1. SPATIAL SETUP
# The 'bbox' from the YAML is converted into a WKT Polygon[cite: 4].
# =======================================================================
raw_bbox = (2.591733, 50.680797, 5.883999, 51.963900)
wkt_poly = sql.bbox2polygon_wkt(raw_bbox)

# =======================================================================
# 2. TAXONOMIC SETUP
# Because 'col_backbone: true' is in the YAML, we use the mapper function 
# to assign these CoL string keys to their appropriate rank columns (like 
# speciesKey or genusKey)[cite: 4].
# =======================================================================
raw_taxa = ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV',
            '3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
mapped_taxa = sql.map_taxonkeys_to_columns(raw_taxa, col_backbone=True)

# =======================================================================
# 3. DATA QUALITY SETUP
# The 'exclude_issues' array is converted into GBIF Hive SQL statements[cite: 4].
# =======================================================================
issue_flags = [
    "hasCoordinate = TRUE",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'ZERO_COORDINATE', TRUE)",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'COORDINATE_OUT_OF_RANGE', TRUE)",
    "NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'COUNTRY_COORDINATE_MISMATCH', TRUE)"
]

# =======================================================================
# 4. AGGREGATION SETUP (API CUBE SPECIFIC)
# Because processing_mode is "api_cube", the orchestrator translates the 
# YAML 'aggregate' metrics into literal SQL expressions[cite: 2].
# =======================================================================
# Group by the temporal dimensions (GBIF auto-groups the spatial grid cell)
sql_group_cols = ["year", "month", "speciesKey"]

# Translate the YAML metrics into SQL aggregate functions
sql_metric_selects = [
    "COUNT(gbifID) AS total_occurrences",       # From method: "count"
    "COUNT(DISTINCT recordedBy) AS total_observers" # From method: "nunique"
]

# =======================================================================
# 5. GENERATE THE SQL QUERY
# Pass all the parameters to the core SQL builder[cite: 4].
# =======================================================================
manual_api_cube_query = sql.generate_query(
    taxonKeys=mapped_taxa,
    
    # Base columns we need to access. 
    columns=["gbifid", "year", "month", "recordedby"],
    
    record_type="occurrence",               
    wkt_polygon=wkt_poly,
    
    year_range=[2004, 2025],
    month_range=[1, 12],
    
    # CRITICAL API CUBE SETTINGS:
    # aggregate=True tells the builder to use GROUP BY instead of listing raw points[cite: 2, 4].
    aggregate=True,                        
    
    # Pass the dynamic group columns and metric select strings we built above[cite: 2, 4].
    sql_group_cols=sql_group_cols,
    sql_metric_selects=sql_metric_selects,
    
    # We must explicitly pass the base grid and integer resolution for the API cube[cite: 2, 4].
    grid="EEA",                             
    grid_resolution=1000,
    
    coordinateUncertainty=1000,
    max_uncertainty=1000,
    issue_flags=issue_flags,
    col_backbone=True
)

print(manual_api_cube_query)

SELECT
    "year",
    "month",
    classificationdetails['7ddf754f-d193-4cc9-b351-99906754a03b']['specieskey'] AS specieskey,
    COUNT(gbifID) AS total_occurrences,
    COUNT(DISTINCT recordedBy) AS total_observers,
    GBIF_EEARGCode(1000, decimalLatitude, decimalLongitude, COALESCE(coordinateUncertaintyInMeters, 1000)) AS eeacellcode
FROM occurrence
WHERE
    decimalLatitude >= 50.680797 AND decimalLatitude <= 51.9639
    AND decimalLongitude >= 2.591733 AND decimalLongitude <= 5.883999
    AND GBIF_Within('POLYGON ((5.883999 50.680797, 5.883999 51.9639, 2.591733 51.9639, 2.591733 50.680797, 5.883999 50.680797))', decimalLatitude, decimalLongitude) = TRUE
    AND COALESCE(coordinateUncertaintyInMeters, 1000) <= 1000
    AND "year" IS NOT NULL AND "month" IS NOT NULL
    AND "year" >= 2004 AND "year" <= 2025
    AND "month" >= 1 AND "month" <= 12
    AND hasCoordinate = TRUE
    AND NOT GBIF_STRINGARRAYCONTAINS(occurrence.issue, 'ZERO_COORDINATE', TRUE)
    AND NOT GBIF_STRINGARRAYC

#### Using the cube recipe

When we request data that is precubed through the GBIF api we solely need to change the processing mode in which the data source operates. 

In [ ]:
import yaml
# Assuming your module is imported as gbif
# import gbif 

cubed_data_recipe = """
base_dir: "./gbif/"
cube_name: "gbif_cube"

export_as:
  format: "zarr"

spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox:
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900

temporal:
  start_year: 2004
  start_month: 1
  end_year: 2025
  end_month: 12

sources:
  gbif:
    query_filters:
      taxon_keys: ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV','3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
      record_type: "presence"
      default_Uncertainty: 1000
      max_uncertainty: 1000
      exclude_issues:
        - "ZERO_COORDINATE"
        - "COORDINATE_OUT_OF_RANGE"
        - "COUNTRY_COORDINATE_MISMATCH" 
    taxonomy:
      col_backbone: true
    columns: ["year", "month", "recordedby"]
    
    # ---------------------------------------------------------
    # Must be 'api_cube' to trigger the GBIF SQL aggregation
    # ---------------------------------------------------------
    processing_mode: "api_cube"
    
    aggregate:
      group_by_columns: ["year", "month", "speciesKey"]
      
      metrics:
        # 1. Total Occurrences (Server-side row count)
        - column: "gbifID" 
          method: "count"
          weighted: false
          rename: "total_occurrences"
          
        # 2. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "total_observers"
"""

# Parse the YAML string into a Python dictionary
parsed_recipe = yaml.safe_load(cubed_data_recipe)

# Pass the parsed dictionary to the generator
cube = gbif.gbif_cube()
cube_query = cube.generate_gbif_query_from_recipe(parsed_recipe)
print(cube_query)

Constructed and resolved master grid key: EEA_1km
Computing safe fetch envelope for target grid 'EEA_1km'...
source_resolution omitted for custom CRS 'EPSG:4326'. Applying default 30 arc-second fallback: 0.008333333333333333
Safe Source Envelope (EPSG:4326): (2.34133, 50.47560, 6.03892, 52.16588)
SELECT
    "year",
    "month",
    classificationdetails['7ddf754f-d193-4cc9-b351-99906754a03b']['specieskey'] AS specieskey,
    COUNT(gbifID) AS total_occurrences,
    COUNT(DISTINCT recordedBy) AS total_observers,
    GBIF_EEARGCode(1000, decimalLatitude, decimalLongitude, COALESCE(coordinateUncertaintyInMeters, 1000)) AS eeacellcode
FROM occurrence
WHERE
    decimalLatitude >= 50.47560476531819 AND decimalLatitude <= 52.16588306988871
    AND decimalLongitude >= 2.34133036280974 AND decimalLongitude <= 6.038920099374839
    AND GBIF_Within('POLYGON ((6.038920099374839 50.47560476531819, 6.038920099374839 52.16588306988871, 2.34133036280974 52.16588306988871, 2.34133036280974 50.4756047653

### Submitting and fetching data

When the query is ready we can submit our data to the GBIF API. However in order to do that the user has to make sure that they have a registered account with GBIF as they need their credentials in order to submit and download. The library either excepts that your credentials are

- stored in the .env where the code is running 
- passed an optional argument to the submit and download function
- Will ask you to enter it in the session in which the code is running in case the two previous methods were not submitted or present

A credentials dictionary has the following structure

```
{
    'GBIF_USER':'yourUserName', 
    'GBIF_MAIL':'yourEmail@provider.com', 
    'GBIF_PWD':'yourPassword'
}
```

In [ ]:
sql.submit_gbif_query(point_query)

✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\tutorials\.env.


INFO:Your sql download key is 0003723-260721160103020


GBIF download submitted successfully. Key: 0003723-260721160103020


'0003723-260721160103020'

In [ ]:
sql.submit_gbif_query(cube_query)

✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\tutorials\.env.


INFO:Your sql download key is 0003752-260721160103020


GBIF download submitted successfully. Key: 0003752-260721160103020


'0003752-260721160103020'

In [ ]:
sql.fetch_gbif_download('0003723-260721160103020', 'gbif/point/')
sql.fetch_gbif_download('0003752-260721160103020', 'gbif/cubed/')

INFO:Download file size: 23248356 bytes


✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\tutorials\.env.
Polling status for download key: 0003723-260721160103020...
Download succeeded!


INFO:On disk at gbif/point//0003723-260721160103020.zip


✅ GBIF credentials are valid.
Credentials stored in C:\Users\niels\Documents\Repositories\BmC\tutorials\.env.
Polling status for download key: 0003752-260721160103020...
Download succeeded!


INFO:Download file size: 8418854 bytes
INFO:On disk at gbif/cubed//0003752-260721160103020.zip


{'path': 'gbif/cubed//0003752-260721160103020.zip',
 'size': 8418854,
 'key': '0003752-260721160103020'}

# `gbif_cube`

We will demonstrate how we can utilize the gbif cubing engine to perform complex vector and grid mathematics to obtain processed vector data that is spatially aligned and harmonized with the raster data produced by the `spatiotemporal_cube` class. In order to control processing we will mainly be focussing on how to utilize the `vector_processing` and `aggregate` sections of the recipe which will be translatable to any vector data source.

We define a base_yaml that describes the taxonomic, spatial and temporal parameters that characterize gbif data and add our `vector_processing` and `aggregate` where needed to it 

In [1]:
from bmc.cube import spatiotemporal, gbif
import yaml
import copy
from pprint import pprint

# =======================================================================
# 1. DEFINE BASE RECIPE
# We define the shared spatial boundaries and query filters once.
# =======================================================================
base_yaml = """
base_dir: "./gbif_test_outputs/"
cube_name: "tutorial_cube"

spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox:
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900

sources:
  gbif:
    query_filters:
      taxon_keys: ['G59D'] # Shortened for brevity
      default_Uncertainty: 1000
    columns: ["year", "month", "recordedby"]
    time_cols: ["year", "month"]
"""
base_recipe = yaml.safe_load(base_yaml)

# Path to your pre-downloaded GBIF zip file
local_zip_path = "./gbif/downloads/0003723-260721160103020.zip"

# Instantiate the spatial pipeline engine
cube = gbif.gbif_cube()

## Point processing

### Raw

In order to process the data without mapping to any of the environmental layers we only need to add the 

```
processing_mode:'raw'
```

to the script. This will export a geopandas dataframe where the points are stored without any grid cell assignment. This is the option that should be followed in case an end user wishes to perform their own propertary gridding algorithm to the gbif point data

In [2]:
print("\n--- Processing RUN 1: Raw Points ---")
recipe_raw = copy.deepcopy(base_recipe)
recipe_raw["sources"]["gbif"]["processing_mode"] = "raw"

pprint(recipe_raw)


--- Processing RUN 1: Raw Points ---
{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'raw',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']}}},
 'spatial': {'bbox': {'lat_max': 51.9639,
                      'lat_min': 50.680797,
                      'long_max': 5.883999,
                      'long_min': 2.591733},
             'target_grid': 'EEA',
             'target_resolution': '1km',
             'use_bbox': True}}


In [3]:
raw_gdf = cube.process_cube(
    recipe=recipe_raw,
    dataset_name="gbif",
    output_filepath="./gbif/01_raw_points.parquet",
    downloaded_filepath=local_zip_path
)


=== Initiating GBIF Generation ===
Processing Mode: RAW
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Raw mode requested. Generating basic Point geometries with no vector transformations.
Bypassing spatial engine. Exporting RAW data to ./gbif/01_raw_points.parquet...


In [4]:
raw_gdf

,gbifid,specieskey,year,month,recordedby,decimallatitude,decimallongitude,coordinateuncertaintyinmeters,geometry
0,4571446352,4HPXM,2022,4,CROIZE Nicolas,50.690280,3.074520,1000.0,POINT (3.07452 50.69028)
1,4571446353,75SVV,2022,4,CROIZE Nicolas,50.690280,3.074520,1000.0,POINT (3.07452 50.69028)
2,4115560010,4F6YZ,2008,5,NaN,51.300000,2.870800,1000.0,POINT (2.8708 51.3)
3,4114262729,75SVV,2012,11,NaN,51.301400,2.465000,1000.0,POINT (2.465 51.3014)
4,4114204757,75SVV,2012,10,NaN,51.486000,2.844100,1000.0,POINT (2.8441 51.486)
...,...,...,...,...,...,...,...,...,...
648472,5087071359,7G3C6,2024,6,Barbara Terhal,50.877832,5.939729,177.0,POINT (5.93973 50.87783)
648473,5176569798,7G3C6,2025,6,Горбунов Максим Сергеевич,50.865380,4.676651,1000.0,POINT (4.67665 50.86538)
648474,5292200124,5B9T3,2025,8,Max Devis,51.333711,5.011200,4.0,POINT (5.0112 51.33371)
648475,5890157589,G59D,2025,10,Saim,51.045357,5.157851,1000.0,POINT (5.15785 51.04536)


The end product is a geopandas dataframe which is only extended by a `geometry` column containing the point representation of the record

### Point data processing

Point data can also be processed so that the resulting geometry is matched to the same raster as all the other layers in the cube will be. In order to perform this matching we extend the recipe with 

```
processing_mode: 'vector'
vector_processing:
    topology: 'point'
    mapping_mode: 'classification
    spatial_method: 'intersect' or 'kdtree'
```

In [2]:
recipe_pts = copy.deepcopy(base_recipe)
recipe_pts["sources"]["gbif"]["processing_mode"] = "vector"
recipe_pts["sources"]["gbif"]["vector_processing"] = {
    "topology": "point",
    "mapping_mode": "classification",
    "spatial_method": "intersect"
}
pprint(recipe_pts)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'vector',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']},
                      'vector_processing': {'mapping_mode': 'classification',
                                            'spatial_method': 'intersect',
                                            'topology': 'point'}}},
 'spatial': {'bbox': {'lat_max': 51.9639,
                      'lat_min': 50.680797,
                      'long_max': 5.883999,
                      'long_min': 2.591733},
             'target_grid': 'EEA',
             'target_resolution': '1km',
             'use_bbox': True}}


In [3]:
pts_gdf = cube.process_cube(
    recipe=recipe_pts,
    dataset_name="gbif",
    output_filepath="./gbif_test_outputs/02_vector_points.parquet",
    downloaded_filepath=local_zip_path
)
pts_gdf


=== Initiating GBIF Generation ===
Processing Mode: VECTOR
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Converting tabular coordinates to POINT geometries in EPSG:4326...
Point generation complete.
Initiating vector geometry sanitization...
Sanitization complete. Final feature count: 648477
Constructed and resolved master grid key: EEA_1km
Extracting padded global bounding box from recipe...
Using explicitly provided target bounding box: (3797404.950727706, 3070408.6245393865, 4039267.786405305, 3232839.0500113894)
Mapping simple points via spatial intersection...
72396 record(s) fell outside the grid extent.
=== Initiating Dynamic Vector QA/QC Profiling ===
=== Dynamic Vector QA/QC Complete ===
Validation report saved to c:\Users\niels\Documents\Repositories\BmC\tutorials\gbif_test_outputs\validation_report\gbif_qa_qc_failures.json
No 'aggregate' block found. Returning vector data as-is.
Exporting final spatial dataset to ./gbif_test_outputs/02_vector_points

,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters,geometry,grid_idx
0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),9649832
1,4571446353,75SVV,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),9649832
2,4115560010,4F6YZ,2008,5,NaN,1000.0,POINT (3824688.806 3156360.289),9373824
3,4114262729,75SVV,2012,11,NaN,1000.0,POINT (3796567.037 3159352.014),<NA>
4,4114204757,75SVV,2012,10,NaN,1000.0,POINT (3824856.865 3177152.153),9289824
...,...,...,...,...,...,...,...,...
648472,5087071359,7G3C6,2024,6,Barbara Terhal,177.0,POINT (4035392.064 3093078.915),9626035
648473,5176569798,7G3C6,2025,6,Горбунов Максим Сергеевич,1000.0,POINT (3946603.835 3097394.021),9609946
648474,5292200124,5B9T3,2025,8,Max Devis,4.0,POINT (3973626.323 3147735.077),9409973
648475,5890157589,G59D,2025,10,Saim,1000.0,POINT (3981704.49 3115027.201),9537981


During processing of the data we take note that the logger returns

```
72396 record(s) fell outside the grid extent.
```

This should not alarm the user when inspecting the log. During data collection we utilize a buffered bounding box to take into account that there exists no 1 to 1 perfect overlap between the different coordinate reference systems. When we map points to the final raster we construct the bounding box in the target crs which is smaller than the fetching bbox and thus some points will become redundant and be removed from the end data product.

## Point cloud processing

Similar to the simple point assignment we can utilize point clouds to take the uncertainty of observation into account. The point clouds represent a Monte Carlo simulation where we generate a set number of randomized points within the possibility space of the geometry. With these point clouds we can either

- Perform grid cell assignment similar to the point geometry. In this case the observation will be assigned to the cell that contains the largest fraction of points from this point cloud
- Calculate fractional weights over multiple cells by counting the number of point in each cell and returning a weight with respect to the total number of points

In either case we must provide arguments in the `vector_processing` that determine the parameters of the simulation 

```
processing_mode: 'vector'
vector_processing:
    topology: 'point_cloud'
    mapping_mode: 'classify' or 'fractional'
    spatial_method: 'intersect' or 'kdtree'
    point_cloud_config:
        n_passes: integer
        distribution: "gaussian" or "uniform"
        random_state: 42
```

The random_state in this config allows us to recreate the point cloud without having to store the entire cloud on disc. Choosing the state to a set number assures consist random sampling

#### Point cloud grid cell assignment

For cell classification we remain with the `mapping:'classification'` and either the centroid based method or the areal intersection method to determine grid cell assignment. Centroid assignment picks the cell that minimizes centroid distance wrt all the points in the cloud and areal intersection chooses the cell populated by most points 

In [5]:
recipe_cloud_discrete = copy.deepcopy(base_recipe)
recipe_cloud_discrete["sources"]["gbif"]["processing_mode"] = "vector"
recipe_cloud_discrete["sources"]["gbif"]["vector_processing"] = {
    "topology": "point_cloud",
    "mapping_mode": "classify",
    "spatial_method": "kdtree", # Snaps cloud to mathematical center
    "topology_config": {
        "point_cloud": {"n_passes": 100, "distribution": "gaussian", "random_seed": 42}
    }
}

pprint(recipe_cloud_discrete)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'vector',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']},
                      'vector_processing': {'mapping_mode': 'classify',
                                            'spatial_method': 'kdtree',
                                            'topology': 'point_cloud',
                                            'topology_config': {'point_cloud': {'distribution': 'gaussian',
                                                                                'n_passes': 100,
                                                                                'random_seed': 42}}}}},
 'spatial': {'bbox': {'lat_max': 51.9639,
                      'lat_min': 50.680797,
                      'long_max': 5.883999,
                      'long

In [3]:
cloud_discrete_gdf = cube.process_cube(
    recipe=recipe_cloud_discrete,
    dataset_name="gbif",
    output_filepath="./gbif_test_outputs/03_cloud_discrete.parquet",
    downloaded_filepath=local_zip_path
)
cloud_discrete_gdf


=== Initiating GBIF Generation ===
Processing Mode: VECTOR
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Converting tabular coordinates to POINT_CLOUD geometries in EPSG:4326...
Delegating to generate_spatial_point_clouds...
Generating strictly 100 point(s) per feature (64,847,700 total points across 648,477 records).
Initiating vector geometry sanitization...
Sanitization complete. Final feature count: 648477
Constructed and resolved master grid key: EEA_1km
Extracting padded global bounding box from recipe...
Using explicitly provided target bounding box: (3797404.950727706, 3070408.6245393865, 4039267.786405305, 3232839.0500113894)
Mapping original source points to determine true home grid cells...
Classifying point clouds via mathematical cloud centroid...
=== Initiating Dynamic Vector QA/QC Profiling ===
=== Dynamic Vector QA/QC Complete ===
Validation report saved to c:\Users\niels\Documents\Repositories\BmC\tutorials\gbif_test_outputs\validation_report\

,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters,geometry,passes,weight_per_point,grid_idx,src_grid_idx
0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),100,0.01,9649832,9649832
1,4571446353,75SVV,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),100,0.01,9649832,9649832
2,4115560010,4F6YZ,2008,5,NaN,1000.0,POINT (3824688.806 3156360.289),100,0.01,9373824,9373824
3,4114262729,75SVV,2012,11,NaN,1000.0,POINT (3796567.037 3159352.014),100,0.01,9361797,9361797
4,4114204757,75SVV,2012,10,NaN,1000.0,POINT (3824856.865 3177152.153),100,0.01,9289824,9289824
...,...,...,...,...,...,...,...,...,...,...,...
648472,5087071359,7G3C6,2024,6,Barbara Terhal,177.0,POINT (4035392.064 3093078.915),100,0.01,9626035,9626035
648473,5176569798,7G3C6,2025,6,Горбунов Максим Сергеевич,1000.0,POINT (3946603.835 3097394.021),100,0.01,9609946,9609946
648474,5292200124,5B9T3,2025,8,Max Devis,4.0,POINT (3973626.323 3147735.077),100,0.01,9409973,9409973
648475,5890157589,G59D,2025,10,Saim,1000.0,POINT (3981704.49 3115027.201),100,0.01,9537981,9537981


#### Point cloud fractional weight generation

When employing the `mapping_mode:fractional` we can only use the `intersect` spatial method (`kdtree` would not make sense here). 

In [2]:
recipe_cloud_fractional = copy.deepcopy(base_recipe)
recipe_cloud_fractional["sources"]["gbif"]["processing_mode"] = "vector"
recipe_cloud_fractional["sources"]["gbif"]["vector_processing"] = {
    "topology": "point_cloud",
    "mapping_mode": "fractional",
    "spatial_method": "intersect", # Shatters cloud across grid cells
    "topology_config": {
        "point_cloud": {"n_passes": 100, "distribution": "gaussian", "random_seed": 42}
    }
}

pprint(recipe_cloud_fractional)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'vector',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']},
                      'vector_processing': {'mapping_mode': 'fractional',
                                            'spatial_method': 'intersect',
                                            'topology': 'point_cloud',
                                            'topology_config': {'point_cloud': {'distribution': 'gaussian',
                                                                                'n_passes': 100,
                                                                                'random_seed': 42}}}}},
 'spatial': {'bbox': {'lat_max': 51.9639,
                      'lat_min': 50.680797,
                      'long_max': 5.883999,
                      

In [3]:
cloud_fractional_gdf = cube.process_cube(
    recipe=recipe_cloud_fractional,
    dataset_name="gbif",
    output_filepath="./gbif_test_outputs/04_cloud_fractional.parquet",
    downloaded_filepath=local_zip_path
)
cloud_fractional_gdf


=== Initiating GBIF Generation ===
Processing Mode: VECTOR
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Converting tabular coordinates to POINT_CLOUD geometries in EPSG:4326...
Delegating to generate_spatial_point_clouds...
Generating strictly 100 point(s) per feature (64,847,700 total points across 648,477 records).
Initiating vector geometry sanitization...
Sanitization complete. Final feature count: 648477
Constructed and resolved master grid key: EEA_1km
Extracting padded global bounding box from recipe...
Using explicitly provided target bounding box: (3797404.950727706, 3070408.6245393865, 4039267.786405305, 3232839.0500113894)
Mapping original source points to determine true home grid cells...
Exploding and intersecting clouds in 65 memory-safe batches...
Calculating fractional probability weights per grid cell...
9845 point cloud(s) have some jittered points falling outside the grid extent; their returned fractions sum to < 1.0.
67882 point cloud(s) f

,grid_idx,src_grid_idx,fraction,src_uid,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters,passes,weight_per_point
0,9645832,9649832,0.04,0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,100,0.01
1,9649831,9649832,0.05,0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,100,0.01
2,9649832,9649832,0.85,0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,100,0.01
3,9653832,9649832,0.05,0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,100,0.01
4,9653833,9649832,0.01,0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,100,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2392334,9537981,9537981,0.46,648475,5890157589,G59D,2025,10,Saim,1000.0,100,0.01
2392335,9537982,9537981,0.10,648475,5890157589,G59D,2025,10,Saim,1000.0,100,0.01
2392336,9541980,9537981,0.01,648475,5890157589,G59D,2025,10,Saim,1000.0,100,0.01
2392337,9541981,9537981,0.37,648475,5890157589,G59D,2025,10,Saim,1000.0,100,0.01


The fractional parquet file is a relational table that stores the source geometry and where this source is located as well. Loading the source_geometries parquet file allows us to link it back to the original data

In [7]:
import geopandas as gpd
# Define the path to your exported source geometries file
file_path = "./gbif_test_outputs/04_cloud_fractional_source_geometries.parquet"
# Load the file directly into a GeoDataFrame
source_gdf = gpd.read_parquet(file_path)
source_gdf

,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters,geometry,passes,weight_per_point,src_uid
0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,POINT (3.07452 50.69028),100,0.01,0
1,4571446353,75SVV,2022,4,CROIZE Nicolas,1000.0,POINT (3.07452 50.69028),100,0.01,1
2,4115560010,4F6YZ,2008,5,NaN,1000.0,POINT (2.8708 51.3),100,0.01,2
3,4114262729,75SVV,2012,11,NaN,1000.0,POINT (2.465 51.3014),100,0.01,3
4,4114204757,75SVV,2012,10,NaN,1000.0,POINT (2.8441 51.486),100,0.01,4
...,...,...,...,...,...,...,...,...,...,...
648472,5087071359,7G3C6,2024,6,Barbara Terhal,177.0,POINT (5.93973 50.87783),100,0.01,648472
648473,5176569798,7G3C6,2025,6,Горбунов Максим Сергеевич,1000.0,POINT (4.67665 50.86538),100,0.01,648473
648474,5292200124,5B9T3,2025,8,Max Devis,4.0,POINT (5.0112 51.33371),100,0.01,648474
648475,5890157589,G59D,2025,10,Saim,1000.0,POINT (5.15785 51.04536),100,0.01,648475


## Polygon processing

### Polygon weight generation

We can also instruct the engine to construct tabular data in which polygon geometries are shattered across the target raster and return the weights associated with each grid cell. For this setting we change `topology:polygon`, `mapping_mode` and `spatial_method` are always `fractional` and `intersect` in this case. The only parameter that can be controlled of significance here is the `topology_config` where we control `quad`. This parameter dictates how many quadrants the constructed geometry should have and determines the smoothness of the constructed polygon

In [2]:
recipe_polygon = copy.deepcopy(base_recipe)
recipe_polygon["sources"]["gbif"]["processing_mode"] = "vector"
recipe_polygon["sources"]["gbif"]["vector_processing"] = {
    "topology": "polygon",
    "mapping_mode": "fractional",
    "spatial_method": "intersect",
    "topology_config": {
        "polygon": {"quad_segs": 8}
    }
}

pprint(recipe_polygon)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'vector',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']},
                      'time_cols': ['year', 'month'],
                      'vector_processing': {'mapping_mode': 'fractional',
                                            'spatial_method': 'intersect',
                                            'topology': 'polygon',
                                            'topology_config': {'polygon': {'quad_segs': 8}}}}},
 'spatial': {'bbox': {'lat_max': 51.9639,
                      'lat_min': 50.680797,
                      'long_max': 5.883999,
                      'long_min': 2.591733},
             'target_grid': 'EEA',
             'target_resolution': '1km',
             'use_bbox': True}}


In [5]:
polygon_gdf = cube.process_cube(
    recipe=recipe_polygon,
    dataset_name="gbif",
    output_filepath="./gbif_test_outputs/05_polygon_buffers.parquet",
    downloaded_filepath=local_zip_path
)
polygon_gdf


=== Initiating GBIF Generation ===
Processing Mode: VECTOR
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Converting tabular coordinates to POLYGON geometries in EPSG:4326...
Applying metric coordinate uncertainty buffers...
Geographic CRS detected. Dynamically mapping records to localized UTM/UPS zones...
Batch-processing polygons within localized metric zones...
Coordinate uncertainty polygon generation complete.
Initiating vector geometry sanitization...
Sanitization complete. Final feature count: 648477
Constructed and resolved master grid key: EEA_1km
Extracting padded global bounding box from recipe...
Using explicitly provided target bounding box: (3797404.950727706, 3070408.6245393865, 4039267.786405305, 3232839.0500113894)
Preparing polygons and calculating baseline continuous metric areas...
Mapping polygon centroids to determine home grid cells...
Executing polygon network fragmentation against template mesh in 65 memory-safe batches...
12875 polygon

,grid_idx,src_uid,centroid_grid_idx,areal_fraction,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters
0,9645831,0,9649832,0.022087,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
1,9645832,0,9649832,0.119299,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
2,9645833,0,9649832,0.011010,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
3,9649831,0,9649832,0.163263,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
4,9649832,0,9649832,0.320108,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
...,...,...,...,...,...,...,...,...,...,...
3249249,9537981,648475,9537981,0.303987,5890157589,G59D,2025,10,Saim,1000.0
3249250,9537982,648475,9537981,0.163058,5890157589,G59D,2025,10,Saim,1000.0
3249251,9541980,648475,9537981,0.042913,5890157589,G59D,2025,10,Saim,1000.0
3249252,9541981,648475,9537981,0.288791,5890157589,G59D,2025,10,Saim,1000.0


## Aggregating dating 

Using the recipe we can also choose to take the processed data from `process_cube` and aggregate it to form a condensed table. The aggregate config blocks follows a general structure where we 

- list the columns that serve as the grouping variables
- metrics that dictate how specific columns should be condensed using standardized pandas function

Aggregation can be perform for both the `classify` and `fractional` outputs. In both cases we standardly choose the `grid_idx` as one of the default columns over which should be aggregated. Below we provide a general example of an aggregate block within the cubing recipe. The aggregate block also allows us to export the unprocessed data beside the aggregated format.

```
aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["year", "month"]
      
      metrics:
        # 1. Expected Occurrences (Sums the geometric fractions)
        - column: "areal_fraction"
          method: "sum"
          weighted: false
          rename: "expected_occurrences"
          
        # 2. Species Count / Richness
        - column: "speciesKey"
          method: "nunique"
          weighted: false
          rename: "species_richness"
          
        # 3. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "distinct_observers"
```

#### Aggregating point data

In [8]:
recipe_cloud_discrete["sources"]["gbif"]["aggregate"] = {
    "group_by_columns": ["specieskey", "year", "month"],
    "metrics": [
        {
            "column": "gbifid",
            "method": "nunique",
            "weighted": False,
            "rename": "observation_count"
        }
    ]
}
pprint(recipe_cloud_discrete)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'aggregate': {'group_by_columns': ['specieskey',
                                                         'year',
                                                         'month'],
                                    'metrics': [{'column': 'gbifid',
                                                 'method': 'nunique',
                                                 'rename': 'observation_count',
                                                 'weighted': False}]},
                      'columns': ['year', 'month', 'recordedby'],
                      'processing_mode': 'vector',
                      'query_filters': {'default_Uncertainty': 1000,
                                        'taxon_keys': ['G59D']},
                      'vector_processing': {'mapping_mode': 'classify',
                                            'spatial_method': 'kdtree',
                                            

In [9]:
# Define the path to your exported source geometries file
file_path = "./gbif_test_outputs/03_cloud_discrete.parquet"

# Load the file directly into a GeoDataFrame
cloud_discrete_gdf = gpd.read_parquet(file_path)
cloud_discrete_gdf

,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters,geometry,passes,weight_per_point,grid_idx,src_grid_idx
0,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),100,0.01,9649832,9649832
1,4571446353,75SVV,2022,4,CROIZE Nicolas,1000.0,POINT (3832439.326 3087420.663),100,0.01,9649832,9649832
2,4115560010,4F6YZ,2008,5,NaN,1000.0,POINT (3824688.806 3156360.289),100,0.01,9373824,9373824
3,4114262729,75SVV,2012,11,NaN,1000.0,POINT (3796567.037 3159352.014),100,0.01,9361797,9361797
4,4114204757,75SVV,2012,10,NaN,1000.0,POINT (3824856.865 3177152.153),100,0.01,9289824,9289824
...,...,...,...,...,...,...,...,...,...,...,...
648472,5087071359,7G3C6,2024,6,Barbara Terhal,177.0,POINT (4035392.064 3093078.915),100,0.01,9626035,9626035
648473,5176569798,7G3C6,2025,6,Горбунов Максим Сергеевич,1000.0,POINT (3946603.835 3097394.021),100,0.01,9609946,9609946
648474,5292200124,5B9T3,2025,8,Max Devis,4.0,POINT (3973626.323 3147735.077),100,0.01,9409973,9409973
648475,5890157589,G59D,2025,10,Saim,1000.0,POINT (3981704.49 3115027.201),100,0.01,9537981,9537981


In [10]:
cube_cloud_classify = cube.aggregate_vector_cube(
    data=cloud_discrete_gdf,            # Your dataframe from the classification run
    recipe=recipe_cloud_discrete, 
    dataset_name="gbif"
)
cube_cloud_classify

Aggregating vector data by dimensions: ['grid_idx', 'specieskey', 'year', 'month']...
Discrete classification detected. Spatial multiplier set to 1.0.
Aggregating 'gbifid' -> 'observation_count' (Method: nunique, Weighted: False)
Vector aggregation complete. Yielded 301610 final cells.


,grid_idx,specieskey,year,month,observation_count
0,9649832,4HPXM,2022,4,6
1,9649832,75SVV,2022,4,3
2,9373824,4F6YZ,2008,5,1
3,9361797,75SVV,2012,11,1
4,9289824,75SVV,2012,10,1
...,...,...,...,...,...
301605,9545874,4HPXM,2025,10,1
301606,9626035,7G3C6,2024,6,1
301607,9609946,7G3C6,2025,6,1
301608,9409973,5B9T3,2025,8,1


#### Aggregating fractional data

In [3]:
recipe_polygon["sources"]["gbif"]["aggregate"] = {
    "group_by_columns": ["specieskey", "year", "month"],
    "export_unaggregated": True,
    "metrics": [
        {
            "column": "gbifid",
            "method": "nunique",
            "weighted": False,
            "rename": "unique_source_records"
        },
        {
            "column": "areal_fraction",
            "method": "sum",
            "weighted": False,  
            "rename": "total_fractional_mass"
        }
    ]
}
pprint(recipe_polygon)

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'sources': {'gbif': {'aggregate': {'export_unaggregated': True,
                                    'group_by_columns': ['specieskey',
                                                         'year',
                                                         'month'],
                                    'metrics': [{'column': 'gbifid',
                                                 'method': 'nunique',
                                                 'rename': 'unique_source_records',
                                                 'weighted': False},
                                                {'column': 'areal_fraction',
                                                 'method': 'sum',
                                                 'rename': 'total_fractional_mass',
                                                 'weighted': False}]},
                      'columns': ['year', 'month', 'recordedby'],
        

In [20]:
import pandas as pd
# Define the path to your exported source geometries file
file_path = "./gbif_test_outputs/05_polygon_buffers.parquet"

# Load the file directly into a GeoDataFrame
polygon_gdf = pd.read_parquet(file_path)
polygon_gdf

,grid_idx,src_uid,centroid_grid_idx,areal_fraction,gbifid,specieskey,year,month,recordedby,coordinateuncertaintyinmeters
0,9645831,0,9649832,0.022087,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
1,9645832,0,9649832,0.119299,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
2,9645833,0,9649832,0.011010,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
3,9649831,0,9649832,0.163263,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
4,9649832,0,9649832,0.320108,4571446352,4HPXM,2022,4,CROIZE Nicolas,1000.0
...,...,...,...,...,...,...,...,...,...,...
3249249,9537981,648475,9537981,0.303987,5890157589,G59D,2025,10,Saim,1000.0
3249250,9537982,648475,9537981,0.163058,5890157589,G59D,2025,10,Saim,1000.0
3249251,9541980,648475,9537981,0.042913,5890157589,G59D,2025,10,Saim,1000.0
3249252,9541981,648475,9537981,0.288791,5890157589,G59D,2025,10,Saim,1000.0


In [22]:
print("\n--- Aggregating Polygon Fractional Data ---")
cube_poly_fractional = cube.aggregate_vector_cube(
    data=polygon_gdf,           # Your dataframe from the polygon run
    recipe=recipe_polygon, 
    dataset_name="gbif"
)
cube_poly_fractional


--- Aggregating Polygon Fractional Data ---
Aggregating vector data by dimensions: ['grid_idx', 'specieskey', 'year', 'month']...
Fractional geometries detected. Spatial multiplier: 'areal_fraction'.
Aggregating 'gbifid' -> 'unique_source_records' (Method: nunique, Weighted: False)
Aggregating 'areal_fraction' -> 'total_fractional_mass' (Method: sum, Weighted: False)
Vector aggregation complete. Yielded 1344400 final cells.


,grid_idx,specieskey,year,month,unique_source_records,total_fractional_mass
0,9645831,4HPXM,2022,4,10,0.679704
1,9645832,4HPXM,2022,4,9,0.989150
2,9645833,4HPXM,2022,4,6,0.066060
3,9649831,4HPXM,2022,4,11,1.731432
4,9649832,4HPXM,2022,4,6,1.920650
...,...,...,...,...,...,...
1344395,9537981,G59D,2025,10,1,0.303987
1344396,9537982,G59D,2025,10,1,0.163058
1344397,9541980,G59D,2025,10,1,0.042913
1344398,9541981,G59D,2025,10,1,0.288791


## Complete pipeline

In the final step we will use the pipeline in its totality where we perform fetching, mapping and aggregating into a single step. 

In [23]:
recipe_cloud_discrete

{'base_dir': './gbif_test_outputs/',
 'cube_name': 'tutorial_cube',
 'spatial': {'target_grid': 'EEA',
  'target_resolution': '1km',
  'use_bbox': True,
  'bbox': {'long_min': 2.591733,
   'long_max': 5.883999,
   'lat_min': 50.680797,
   'lat_max': 51.9639}},
 'sources': {'gbif': {'query_filters': {'taxon_keys': ['G59D'],
    'default_Uncertainty': 1000},
   'columns': ['year', 'month', 'recordedby'],
   'processing_mode': 'vector',
   'vector_processing': {'topology': 'point_cloud',
    'mapping_mode': 'classify',
    'spatial_method': 'kdtree',
    'topology_config': {'point_cloud': {'n_passes': 100,
      'distribution': 'gaussian',
      'random_seed': 42}}},
   'aggregate': {'group_by_columns': ['specieskey', 'year', 'month'],
    'metrics': [{'column': 'gbifid',
      'method': 'nunique',
      'weighted': False,
      'rename': 'observation_count'}]}}}}

In [4]:
pc_gdf= cube.process_cube(recipe_cloud_discrete, "gbif", output_filepath="./gbif_test_outputs/06_pointcloud_final.parquet",
    downloaded_filepath=local_zip_path)
pc_gdf

NameError: name 'recipe_cloud_discrete' is not defined

In [4]:
pc_gdf= cube.process_cube(recipe_polygon, "gbif", output_filepath="./gbif_test_outputs/07_polygon_final.parquet",
    downloaded_filepath=local_zip_path)
pc_gdf


=== Initiating GBIF Generation ===
Processing Mode: VECTOR
Output Directory: ./gbif_test_outputs/tutorial_cube\gbif
Loading GBIF data from ./gbif/downloads/0003723-260721160103020.zip...
Converting tabular coordinates to POLYGON geometries in EPSG:4326...
Applying metric coordinate uncertainty buffers...
Geographic CRS detected. Dynamically mapping records to localized UTM/UPS zones...
Batch-processing polygons within localized metric zones...
Coordinate uncertainty polygon generation complete.
Initiating vector geometry sanitization...
Sanitization complete. Final feature count: 648477
Constructed and resolved master grid key: EEA_1km
Extracting padded global bounding box from recipe...
Using explicitly provided target bounding box: (3797404.950727706, 3070408.6245393865, 4039267.786405305, 3232839.0500113894)
Preparing polygons and calculating baseline continuous metric areas...
Mapping polygon centroids to determine home grid cells...
Executing polygon network fragmentation against

<Item id=tutorial_cube_gbif_20260724130407>